In [ ]:
from pathlib import Path
import sys

import polars as pl
from IPython.display import display


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Unable to locate the repository root from the notebook working directory.")


REPO_ROOT = _find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

CACHE_ROOT = REPO_ROOT / "cache"

from wind_datasets.models import TaskSpec
from wind_datasets.persistence import PersistenceExperimentResult, run_persistence_experiment

PERSISTENCE_TASK = TaskSpec.next_6h_from_24h()


def _round_metrics(frame: pl.DataFrame) -> pl.DataFrame:
    rounded_columns = [column for column in ["mae_kw", "rmse_kw", "mae_pu", "rmse_pu"] if column in frame.columns]
    if not rounded_columns:
        return frame
    return frame.with_columns([pl.col(column).round(3) for column in rounded_columns])


def display_persistence_result(result: PersistenceExperimentResult) -> PersistenceExperimentResult:
    summary = _round_metrics(result.summary).with_columns(
        [
            pl.col("start_timestamp").dt.strftime("%Y-%m-%d %H:%M:%S"),
            pl.col("end_timestamp").dt.strftime("%Y-%m-%d %H:%M:%S"),
        ]
    )
    per_horizon = _round_metrics(result.per_horizon.sort("horizon_step"))
    per_turbine = _round_metrics(result.per_turbine)
    display(summary)
    display(per_horizon)
    display(per_turbine)
    return result


In [ ]:
# kelmarsh
kelmarsh_result = run_persistence_experiment(
    "kelmarsh",
    cache_root=CACHE_ROOT,
    task_spec=PERSISTENCE_TASK,
)
display_persistence_result(kelmarsh_result)


In [ ]:
# penmanshiel
penmanshiel_result = run_persistence_experiment(
    "penmanshiel",
    cache_root=CACHE_ROOT,
    task_spec=PERSISTENCE_TASK,
)
display_persistence_result(penmanshiel_result)


In [ ]:
# hill_of_towie
hill_of_towie_result = run_persistence_experiment(
    "hill_of_towie",
    cache_root=CACHE_ROOT,
    task_spec=PERSISTENCE_TASK,
)
display_persistence_result(hill_of_towie_result)


In [ ]:
# sdwpf_kddcup
sdwpf_kddcup_result = run_persistence_experiment(
    "sdwpf_kddcup",
    cache_root=CACHE_ROOT,
    quality_profile="official_v1",
    task_spec=PERSISTENCE_TASK,
)
display_persistence_result(sdwpf_kddcup_result)
